# Week 6: Unsupervised Learning & Clustering
## Notebook 11: Gaussian Random Projection & Johnson-Lindenstrauss Lemma

### 1. Conceptual Intuition & Theoretical Background
When feature dimensions $D$ are massive (e.g. $D = 100,000$ in DNA microarrays or vocabulary n-grams), computing SVD/PCA covariance matrices takes $O(N D^2)$, causing memory crashes.

**Johnson-Lindenstrauss (JL) Lemma (1984):**
Any set of $N$ points in high-dimensional Euclidean space $\mathbb{R}^D$ can be projected into a lower-dimensional space $\mathbb{R}^k$ such that all pairwise Euclidean distances are preserved within a factor of $1 \pm \epsilon$:
$$(1 - \epsilon) \|u - v\|^2 \le \|f(u) - f(v)\|^2 \le (1 + \epsilon) \|u - v\|^2$$
Where the target dimension $k$ depends **only on the number of samples $N$ and error tolerance $\epsilon$, completely independent of original dimension $D$**:
$$k \ge \frac{4 \ln(N)}{\frac{\epsilon^2}{2} - \frac{\epsilon^3}{3}}$$

A **Gaussian Random Matrix** $R \in \mathbb{R}^{D \times k}$ drawn from:
$$R_{ij} \sim \mathcal{N}\left(0, \frac{1}{k}\right)$$
Preserves geometry without learning or fitting!

---

### 2. Scratch Python Implementation: Random Projection Engine

In [ ]:
import numpy as np

def johnson_lindenstrauss_min_dim(n_samples, eps=0.2):
    """Computes minimum target dimension k required by JL lemma."""
    numerator = 4.0 * np.log(n_samples)
    denominator = (eps ** 2) / 2.0 - (eps ** 3) / 3.0
    return int(np.ceil(numerator / denominator))

class GaussianRandomProjectionScratch:
    def __init__(self, n_components):
        self.k = n_components
        self.R = None
        
    def fit_transform(self, X):
        N, D = X.shape
        # Draw projection matrix from N(0, 1/k)
        np.random.seed(42)
        self.R = np.random.normal(0.0, 1.0 / np.sqrt(self.k), (D, self.k))
        # Zero-training cost linear projection: Y = X @ R
        return np.dot(X, self.R)

# Demonstration: High-Dimensional Geometry Preservation
# Project 1,000-dimensional points down to lower dimensions
N = 200
D_original = 1000
eps = 0.25

k_theory = johnson_lindenstrauss_min_dim(N, eps=eps)
k_target = 300 # Practical dimension

X_high = np.random.normal(0, 1.0, (N, D_original))

grp = GaussianRandomProjectionScratch(n_components=k_target)
X_projected = grp.fit_transform(X_high)

# Sample 500 random pairs to evaluate pairwise distance distortion
idx_i = np.random.randint(0, N, 500)
idx_j = np.random.randint(0, N, 500)
mask = idx_i != idx_j
idx_i, idx_j = idx_i[mask], idx_j[mask]

orig_dists_sq = np.sum((X_high[idx_i] - X_high[idx_j]) ** 2, axis=1)
proj_dists_sq = np.sum((X_projected[idx_i] - X_projected[idx_j]) ** 2, axis=1)

distortion_ratios = proj_dists_sq / orig_dists_sq
max_distortion = np.max(np.abs(distortion_ratios - 1.0))

print("--- Johnson-Lindenstrauss Random Projection ---")
print(f"Original Dimension D:          {D_original}")
print(f"Target Compressed Dimension k: {k_target} (Compressed by {(1 - k_target/D_original)*100:.1f}%)")
print(f"Theoretical JL Min Dimension:  {k_theory} (for eps={eps})")
print(f"Observed Max Distortion:       {max_distortion * 100:.1f}%")
print(f"Mean Pairwise Distortion:      {np.mean(np.abs(distortion_ratios - 1.0)) * 100:.1f}%")

### 3. Step-by-Step Code Walkthrough
1. **Dimension Bound ($k$):** Scales logarithmically with $N$, independent of initial feature width $D$.
2. **Scaling Factor ($1/\sqrt{k}$):** Preserves expected squared Euclidean norm: $\mathbb{E}[\|x R\|^2] = \|x\|^2$.
3. **Data-Agnostic Projection:** Requires zero iterative gradient descent or eigendecomposition.

---

### 4. Industry Applications & Trade-offs
- **Vector Search Pre-filtering (FAISS):** Compressing 1536-dimensional OpenAI embeddings to 128 dimensions before computing exact distances.
- **Trade-off:** Random projections are not invertible and cannot reconstruct original feature columns.

---

### 5. Top Interview Questions & Answers
**Q1: Why does Johnson-Lindenstrauss state that target dimension $k$ is independent of original dimension $D$?**
> **Answer:** High-dimensional unit spheres have concentrated surface measure. Almost all random vectors in high dimensions are approximately orthogonal, meaning random projections preserve inner products and distances with high probability.

**Q2: What is the Sparse Random Projection (Achlioptas) variant?**
> **Answer:** Replaces Gaussian distributions with ternary discrete weights $\{-1, 0, 1\}$ where zero occurs with probability $1 - 1/s$. This speeds up matrix multiplication by $3\times$ using sparse pointer additions.